# 03 - Evaluating the Assistant

Notebook 02 tested retrieval on its own. This notebook tests the full assistant: retriever, prompt and LLM together.

I run the same 15 test questions and check two things:

- **Does it answer or refuse correctly?** In-scope questions should get a real answer, and out-of-scope questions should get the fallback message. This is checked automatically.
- **Is each answer grounded?** Everything an answer says should come from its FAQ. I check this by reading each answer next to its FAQ.

This notebook calls the OpenAI API (15 questions, a few cents in total), so it needs `OPENAI_API_KEY` in `.env`.

## Setup

The chain comes from `src/rag_chain.py`. I use `build_rag_chain()`, the version without memory, so each question is answered on its own.

In [26]:
# reload src/ files automatically whenever they change
%load_ext autoreload
%autoreload 2

# make the project root importable (this notebook is in notebooks/)
import sys
sys.path.append("..")

# project code (must come after sys.path.append)
from src import config
from src.rag_chain import build_rag_chain
from src.ingest import load_chunks

# packages
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


True

In [10]:
chain = build_rag_chain()

## Test questions

These are the same 15 questions as notebook 02: 10 in-scope questions with the FAQ id that should answer them, and 5 out-of-scope questions the FAQ can't answer.

In [7]:
test_questions = [
    # in-scope: (question, expected FAQ id)
    ("What hours are you open on Easter?", "0003"),
    ("Can I return shoes I bought?", "0013"),
    ("I want to cancel my order", "0028"),
    ("Do I have to tip the driver?", "0009"),
    ("Can I pay cash for a delivery order?", "0016"),
    ("Will you match Target's prices?", "0018"),
    ("How long until my refund shows up?", "0014"),
    ("Is there a minimum spend for delivery?", "0007"),
    ("Do I need ID to buy beer?", "0022"),
    ("Are you hiring?", "0032"),

    # out-of-scope: not answered by the FAQ
    ("What is a baby dolphin called?", None),
    ("Not sure what I'm looking for.", None),
    ("Do you sell gluten-free bread?", None),
    ("Do you have a pharmacy?", None),
    ("What's the weather tomorrow?", None),
]

## One question first

Before running all 15 questions, I test one to check the chain works and to see what the output looks like. I use an out-of-scope question: the retriever should return no chunks, so the context is empty and the prompt's fallback rule should apply.

In [8]:
question, expected_id = test_questions[10]

print(question)

What is a baby dolphin called?


In [11]:
response = chain.invoke({"input": question, "history": []})
print(response.content)

No relevant docs were retrieved using the relevance score threshold 0.33


I don’t have that information in the provided context. Please email human@abc-grocery.com and they will be glad to assist you!


The threshold removed every chunk, so the LLM received an empty context, and it replied with the fallback message word for word. It didn't use its own knowledge to answer the question, which is the behaviour the grounding rules ask for.

## Run all 15 questions

Each question is sent with an empty history (`"history": []`), so earlier answers can't affect later ones. For each one I save the question, the expected FAQ id, the answer, and whether the answer was the fallback message.

To spot the fallback, I check for the email address `human@abc-grocery.com` rather than the whole sentence.

In [14]:
results = []

for question, expected_id in test_questions:
    response = chain.invoke({"input": question, "history": []})
    answer = response.content

    results.append({
        "question": question,
        "expected_id": expected_id,
        "answer": answer,
        "gave_fallback": "human@abc-grocery.com" in answer
    })


No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33
No relevant docs were retrieved using the relevance score threshold 0.33


In [17]:
pd.set_option("display.max_colwidth", None)
results_df = pd.DataFrame(results)

In [18]:
results_df

,question,expected_id,answer,gave_fallback
0,What hours are you open on Easter?,0003,Holiday hours vary by location. Please check the Store Locator for your store’s exact Easter hours. Most locations are normally open 7am–10pm.,False
1,Can I return shoes I bought?,0013,"Yes—shoes can be returned within 30 days with a receipt if they’re unworn and have the original tags/packaging. You also get one free size exchange within 30 days if the item is in resellable condition. Note that final-sale/clearance items may be ineligible, so please check your receipt.",False
2,I want to cancel my order,0028,"You can cancel until 1 hour before your delivery window. Go to Orders > Upcoming, select the order, and choose Cancel. If you’re within 1 hour of the delivery slot, changes may not be possible.",False
3,Do I have to tip the driver?,0009,"No—tips are optional but appreciated. You can add a tip at checkout or within 24 hours after delivery in Order History, and 100% of tips go to your driver.",False
4,Can I pay cash for a delivery order?,0016,No—cash isn’t accepted for delivery orders.,False
5,Will you match Target's prices?,0018,"No. We don’t match competitor prices (including Target). We honor our own advertised prices in-store and in the app, and prices/promotions may vary by store and online.",False
6,How long until my refund shows up?,0014,"Refunds to your original payment method typically appear within 3–5 business days after processing. If it was a gift card, the refund goes back to the original card.",False
7,Is there a minimum spend for delivery?,0007,"Yes. The minimum order subtotal for delivery is $35 before taxes, fees, and tips. Delivery Club waives the $10 delivery fee, but the $35 minimum still applies.",False
8,Do I need ID to buy beer?,0022,"Yes. A valid government-issued photo ID is required at handoff or pickup. The name on the order must match your ID, and drivers cannot leave alcohol unattended.",False
9,Are you hiring?,0032,Please check our current openings on our website: go to www.abc-grocery.com and click Careers > Work at ABC Grocery. We’d love to have you onboard working with us!,False


## Automatic checks

- **In-scope answered:** how many in-scope questions got a real answer instead of the fallback.
- **Out-of-scope refused:** how many out-of-scope questions got the fallback.

In [25]:
in_scope = results_df[results_df["expected_id"].notna()]
out_of_scope = results_df[results_df["expected_id"].isna()]

in_scope_answered = (in_scope["gave_fallback"] == False).sum()

out_of_scope_refused = out_of_scope["gave_fallback"].sum()

print(f"In-scope answered:     {in_scope_answered} / {len(in_scope)}")
print(f"Out-of-scope refused:  {out_of_scope_refused} / {len(out_of_scope)}")

In-scope answered:     10 / 10
Out-of-scope refused:  5 / 5


All 10 in-scope questions were answered and all 5 out-of-scope questions were refused. The two layers work together: the threshold removes every chunk for out-of-scope questions, and the prompt tells the LLM to use the fallback when the context is empty.

This only shows that the assistant answered, not that the answers are correct, so next I read each one.

In [28]:
in_scope

,question,expected_id,answer,gave_fallback
0,What hours are you open on Easter?,0003,Holiday hours vary by location. Please check the Store Locator for your store’s exact Easter hours. Most locations are normally open 7am–10pm.,False
1,Can I return shoes I bought?,0013,"Yes—shoes can be returned within 30 days with a receipt if they’re unworn and have the original tags/packaging. You also get one free size exchange within 30 days if the item is in resellable condition. Note that final-sale/clearance items may be ineligible, so please check your receipt.",False
2,I want to cancel my order,0028,"You can cancel until 1 hour before your delivery window. Go to Orders > Upcoming, select the order, and choose Cancel. If you’re within 1 hour of the delivery slot, changes may not be possible.",False
3,Do I have to tip the driver?,0009,"No—tips are optional but appreciated. You can add a tip at checkout or within 24 hours after delivery in Order History, and 100% of tips go to your driver.",False
4,Can I pay cash for a delivery order?,0016,No—cash isn’t accepted for delivery orders.,False
5,Will you match Target's prices?,0018,"No. We don’t match competitor prices (including Target). We honor our own advertised prices in-store and in the app, and prices/promotions may vary by store and online.",False
6,How long until my refund shows up?,0014,"Refunds to your original payment method typically appear within 3–5 business days after processing. If it was a gift card, the refund goes back to the original card.",False
7,Is there a minimum spend for delivery?,0007,"Yes. The minimum order subtotal for delivery is $35 before taxes, fees, and tips. Delivery Club waives the $10 delivery fee, but the $35 minimum still applies.",False
8,Do I need ID to buy beer?,0022,"Yes. A valid government-issued photo ID is required at handoff or pickup. The name on the order must match your ID, and drivers cannot leave alcohol unattended.",False
9,Are you hiring?,0032,Please check our current openings on our website: go to www.abc-grocery.com and click Careers > Work at ABC Grocery. We’d love to have you onboard working with us!,False


In [27]:
faq_text = {}

for chunk in load_chunks():
    faq_text[chunk.metadata["id"]] = chunk.page_content

for row in in_scope.itertuples():
    print(f"--- {row.expected_id}: {row.question}")
    print("FAQ:   ", faq_text[row.expected_id])
    print("ANSWER:", row.answer)
    print()

--- 0003: What hours are you open on Easter?
FAQ:    Q: What are your store hours? Do they change on holidays?  
A: Most locations are open 7am–10pm daily. Holiday hours may differ by location—check the Store Locator for exact opening times on specific dates (e.g., Thanksgiving, Christmas, New Year’s Day).
ANSWER: Holiday hours vary by location. Please check the Store Locator for your store’s exact Easter hours. Most locations are normally open 7am–10pm.

--- 0013: Can I return shoes I bought?
FAQ:    Q: How do returns and exchanges work for clothes and shoes?  
A: Clothing and footwear can be returned within 30 days with a receipt if unworn and with original tags/packaging. One free size exchange is allowed within 30 days if the item is in resellable condition. Final-sale/clearance items may be ineligible—check the receipt.
ANSWER: Yes—shoes can be returned within 30 days with a receipt if they’re unworn and have the original tags/packaging. You also get one free size exchange within 

## Manual review

I print each in-scope answer next to the FAQ it should come from, and check whether the answer says anything the FAQ doesn't. With 10 answers, reading them myself is the most reliable check.

| Id | Question | Grounded? | Note |
|---|---|---|---|
| 0003 | Easter hours | Yes | Applies the FAQ's holiday hours to Easter, which is the customer's word. The FAQ's examples don't include Easter, but no new facts are added |
| 0013 | Return shoes | Yes | All details match: 30 days, receipt, unworn with tags, one free size exchange, final-sale items may be ineligible |
| 0028 | Cancel order | Yes | Matches the 1-hour cutoff and the Orders > Upcoming steps. Leaves out Edit and Reschedule, which weren't asked about |
| 0009 | Tip the driver | Yes | Turns "optional" into a direct "No" to the question. Other details match |
| 0016 | Pay cash for delivery | Yes | Matches: cash isn't accepted for delivery orders. Short, but answers the question |
| 0018 | Match Target's prices | Yes | Applies "no competitor price matching" to Target, the customer's word. Other details match |
| 0014 | Refund timing | Yes | Matches: 3–5 business days, and gift card refunds go back to the original card |
| 0007 | Minimum spend | Yes | Matches the $35 minimum, and that Delivery Club removes the $10 fee but not the minimum |
| 0022 | ID for beer | Yes | Applies "age-restricted items" to alcohol, the customer's word. Other details match |
| 0032 | Are you hiring? | Mostly | "Check our current openings" suggests there are open jobs, which the FAQ doesn't say. The careers link is correct. It rightly doesn't answer yes or no |

Rule I used: reusing the customer's words to apply a general answer to their case (Easter, Target, alcohol) is fine. Adding information that is in neither the question nor the FAQ is not.

Only 0032 added something: "current openings" implies there are jobs available, which the FAQ doesn't say. It's a small addition, but it's the kind of detail the grounding rules are meant to prevent.

## Summary

| Check | Result |
|---|---|
| In-scope questions answered | 10 / 10 |
| Out-of-scope questions refused | 5 / 5 |
| In-scope answers fully grounded | 9 / 10 |
| In-scope answers with wrong facts | 0 / 10 |

- **Refusals worked every time.** For all 5 out-of-scope questions, including the shop-like "gluten-free bread", the assistant gave the fallback word for word instead of answering from its own knowledge.
- **9 of 10 answers were fully grounded.** The one exception (0032, hiring) added "current openings", which suggests there are jobs available. The FAQ doesn't say that. It's small, but it shows the model can add details beyond the context.
- **The extra chunks didn't confuse it.** Several questions had up to 4 chunks pass the threshold, such as the delivery windows FAQ for the Easter question, but each answer used only the relevant FAQ.
- **Answers vary between runs.** The Easter answer here is worded differently from the one in the `rag_chain.py` test, even with `temperature=0`. These results come from one run, so a single run isn't proof that every run will behave the same.
- **The test set is small.** 15 questions show the assistant works on common cases, but they don't cover everything, such as questions that need two FAQs, or follow-up questions.

**Next:** test conversation memory in notebook 04, including whether follow-up questions like "how much does *it* cost?" retrieve the right FAQ.